# F3-M00: Índice Fase 3 - Feature Engineering

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 Objetivo

Guía general de Fase 3: Feature Engineering. Índice de módulos y transformación del dataset de **alumno×curso** a **expediente único** listo para modelado ML.

## 📥 Entrada / 📤 Salida

| Tipo | Archivo | Granularidad | Registros |
|------|---------|--------------|----------|
| 📥 Entrada | `02_processed/df_alumno.parquet` | alumno × curso | ~109.568 |
| 📤 Salida | `03_features/df_expediente_features.parquet` | expediente único | ~33.621 |
| 📤 Salida | `automl/df_exp_automl_target_final.parquet` | expediente único | ~33.621 |

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings('ignore')

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'src').exists():
            break
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import (
    RUTA_HTML, RUTA_PROCESSED, RUTA_FEATURES, RUTA_AUTOML,
    info_entorno
)
from src.utils import crear_directorios
from src.html import (
    generar_kpis_html,
    generar_tarjetas_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_base_html

RUTA_FASE3 = RUTA_HTML / 'fase3'
crear_directorios([RUTA_FASE3])

info_entorno()

✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: LEER DATOS REALES DEL PROYECTO (dinámico, sin hardcodes)
# ============================================================================
# Usa RUTA_FEATURES de src.config (= data/03_features/)
# Si el parquet no existe aún (primera ejecución), usa fallback.
# ============================================================================

ruta_parquet = RUTA_FEATURES / 'df_expediente_features.parquet'

if ruta_parquet.exists():
    df_expediente = pd.read_parquet(ruta_parquet)
    N_EXPEDIENTES_REAL = len(df_expediente)
    N_FEATURES_REAL = len(df_expediente.columns)
    del df_expediente
    print(f'✓ Leído: {ruta_parquet.name} ({N_EXPEDIENTES_REAL:,} filas × {N_FEATURES_REAL} cols)')
else:
    N_EXPEDIENTES_REAL = 33621
    N_FEATURES_REAL = 43
    print(f'⚠️  Archivo no encontrado (se generará al ejecutar M01→M05)')
    print(f'   Usando valores por defecto: {N_EXPEDIENTES_REAL:,} expedientes, {N_FEATURES_REAL} features')

✓ Leído: df_expediente_features.parquet (33,621 filas × 50 cols)


In [3]:
# ============================================================================
# CELDA 3: DATOS DE LA PÁGINA (dinámicos)
# ============================================================================

KPIS = [
    {'valor': f'{N_EXPEDIENTES_REAL:,}', 'titulo': 'Expedientes'},
    {'valor': '6', 'titulo': 'Módulos'},
    {'valor': f'+{N_FEATURES_REAL}', 'titulo': 'Features'},
    {'valor': '2021', 'titulo': 'Año referencia'},
]

MODULOS = [
    {
        'archivo': 'm01_validacion.html',
        'emoji': '✅',
        'titulo': 'M01: Validación y Limpieza',
        'desc': 'Verificación de datos, corrección de tipos, eliminación de campos redundantes (P1-P7).',
        'color': '#3182ce'
    },
    {
        'archivo': 'm02_agregacion.html',
        'emoji': '📊',
        'titulo': 'M02: Agregación',
        'desc': '1 fila por expediente único. Longitudinal → Cross-sectional.',
        'color': '#38a169'
    },
    {
        'archivo': 'm03_features.html',
        'emoji': '⚙️',
        'titulo': 'M03: Features',
        'desc': 'Temporal, gaps, progreso académico, estabilidad, velocidad.',
        'color': '#805ad5'
    },
    {
        'archivo': 'm04_encoding.html',
        'emoji': '🔧',
        'titulo': 'M04: Encoding Categóricas',
        'desc': 'M04a: 100% numérico (AutoML). M04b: Mixto (EDA + CatBoost).',
        'color': '#ed8936'
    },
    {
        'archivo': 'm05_target_export.html',
        'emoji': '🎯',
        'titulo': 'M05: Target + Export',
        'desc': 'Calcular abandono (+ nota media/año para segundo estudio). Exportar a automl/ y 04_eda/.',
        'color': '#e53e3e'
    },
]

print(f'✅ Configuración: {len(KPIS)} KPIs, {len(MODULOS)} módulos')

✅ Configuración: 4 KPIs, 5 módulos


In [4]:
# ============================================================================
# CELDA 4: GENERAR HTML
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa="fase3",
    modulo_activo="indice"
)

seccion_resumen = generar_seccion_html(
    titulo="Resumen de la Fase",
    contenido=f"""
        <p>
            Transformación del dataset de <strong>{N_EXPEDIENTES_REAL:,} expedientes</strong> 
            a formato listo para modelado ML.
        </p>
        <p>
            {len(MODULOS)} módulos: validación (con correcciones P1-P7), agregación, ingeniería de features, 
            encoding de categóricas, cálculo del target abandono y export final.
        </p>
        <div style=\"background:#EBF8FF; padding:15px; border-radius:8px; margin-top:15px; border-left:4px solid #3182ce;\">
            <strong>🎯 Objetivo:</strong> Crear dataset listo para AutoML (Fase AutoML) y EDA Final (Fase 4).
        </div>
        <div style=\"background:#F0FFF4; padding:15px; border-radius:8px; margin-top:10px; border-left:4px solid #38a169;\">
            <strong>📁 Salidas:</strong><br>
            • <code>data/03_features/</code> — Parquets intermedios (features, encoding)<br>
            • <code>data/automl/</code> — Dataset 100% numérico con target para AutoML<br>
            • <code>data/04_eda/</code> — Dataset mixto con target para EDA + Modelado
        </div>
    """,
    icono="⚙️"
)

kpis_html = generar_kpis_html(KPIS)
seccion_kpis = generar_seccion_html(
    titulo="Métricas",
    contenido=kpis_html,
    icono="📊"
)

modulos_html = generar_tarjetas_html([
    {
        "titulo": m["titulo"],
        "descripcion": m["desc"],
        "emoji": m["emoji"],
        "link": m["archivo"],
        "link_texto": "Abrir módulo →",
        "color": m["color"]
    }
    for m in MODULOS
])
seccion_modulos = generar_seccion_html(
    titulo="Módulos",
    contenido=modulos_html,
    icono="📦"
)

contenido_html = seccion_resumen + seccion_kpis + seccion_modulos

html_completo = render_base_html(
    titulo="🔧 Fase 3: Feature Engineering",
    icono="🔧",
    subtitulo="Ingeniería de Variables y Preparación para Modelado",
    nav_fases=nav_fases_html,
    nav_modulos=nav_modulos_html,
    contenido=contenido_html,
    notebook_nombre='f3_m00_indice.ipynb',
    notebook_carpeta='fase3_features'
)

ruta_html = RUTA_FASE3 / "fase3_index.html"
guardar_html(html_completo, ruta_html)

print(f"\n✅ HTML generado: {ruta_html}")

GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\fase3_index.html

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\fase3_index.html


In [5]:
# ============================================================================
# CELDA 5: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F3-M00 COMPLETADO')
print('=' * 60)
print(f'\n📋 Archivo: {ruta_html}')
print(f'📊 Contenido: {len(KPIS)} KPIs, {len(MODULOS)} módulos')
print(f'📌 Expedientes (dinámico): {N_EXPEDIENTES_REAL:,}')
print(f'📁 Features guardadas en: {RUTA_FEATURES}')
print(f'📁 AutoML guardado en: {RUTA_AUTOML}')
print(f'\n🎯 Siguiente: f3_m01_validacion.ipynb')
print('=' * 60)


✅ F3-M00 COMPLETADO

📋 Archivo: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\fase3_index.html
📊 Contenido: 4 KPIs, 5 módulos
📌 Expedientes (dinámico): 33,621
📁 Features guardadas en: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
📁 AutoML guardado en: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl

🎯 Siguiente: f3_m01_validacion.ipynb
